# Asset Class Trend Following 策略回測與最佳化 (2024)

本筆記本實作了基於 SMA 與 ROC 的動能策略，包含：
1. 資料清理與前處理
2. 核心回測引擎 (含持股保留規則)
3. 停損機制 (最高價回落與均線停損)
4. 螞蟻演算法 (ACO) 參數最佳化
5. 參數高原表生成
6. 產出詳細 Excel 報告

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xlsxwriter
import os

import warnings
warnings.filterwarnings('ignore')

# 設定顯示中文
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

## 1. 資料讀取與前處理
從 `個股1.xlsx` 讀取資料，設定日期索引並補齊缺失值。

In [ ]:

def load_and_clean_data(filepath):
    df = pd.read_excel(filepath, header=[0, 1])
    # 第二欄為日期
    date_col = df.columns[1]
    df.set_index(date_col, inplace=True)
    df.index = pd.to_datetime(df.index)
    # 移除標籤列
    df.drop(df.columns[0], axis=1, inplace=True)
    # 缺失值填補：forward fill (中途暫停) 與 backward fill (初期未上市)
    df = df.ffill().bfill()
    return df

df = load_and_clean_data('個股1.xlsx')
print(f"資料讀取完成，標的數量: {df.shape[1]}, 交易日天數: {df.shape[0]}")
df.head()


## 2. 回測引擎實作
實作包含 T+1 執行、再平衡週期、持股保留以及停損機制的邏輯。

In [ ]:

def run_backtest(df, sma_len, roc_len, stop_loss_type, stop_loss_val, rb_period=5, rb_offset=0, initial_capital=30_000_000):
    prices = df.copy()
    tickers = prices.columns.get_level_values(0)
    names = prices.columns.get_level_values(1)
    ticker_to_name = dict(zip(tickers, names))
    prices.columns = tickers

    sma = prices.rolling(int(sma_len)).mean()
    roc = prices.pct_change(int(roc_len))

    dates = prices.index
    n_days = len(dates)

    cash = initial_capital
    holdings = {} # ticker -> {'shares': float, 'max_price': float, 'entry_date': date, 'entry_price': float}

    equity = pd.Series(index=dates, dtype=float)
    trade_log = []
    holdings_log = []

    pending_trades = []

    start_idx = max(int(sma_len), int(roc_len))
    if start_idx >= n_days:
        return pd.Series([initial_capital]*n_days, index=dates), [], []

    for i in range(start_idx, n_days):
        curr_date = dates[i]
        curr_prices = prices.iloc[i]

        # 1. 執行掛單 (T+1 收盤)
        if pending_trades:
            sells = [t for t in pending_trades if t['type'] == 'sell']
            buys = [t for t in pending_trades if t['type'] == 'buy']

            for trade in sells:
                ticker = trade['ticker']
                p = curr_prices[ticker]
                shares = trade['shares']
                cash += shares * p
                entry_info = holdings.pop(ticker, None)
                if entry_info:
                    trade_log.append({
                        'Date': curr_date, 'Ticker': ticker, 'Name': ticker_to_name[ticker],
                        'Type': 'Sell', 'Price': p, 'Shares': shares,
                        'Reason': trade['reason'], 'Entry Date': entry_info['entry_date'],
                        'Entry Price': entry_info['entry_price'],
                        'Return': (p / entry_info['entry_price']) - 1 if entry_info['entry_price'] != 0 else 0
                    })

            for trade in buys:
                ticker = trade['ticker']
                p = curr_prices[ticker]
                if p > 0:
                    amount = min(trade['amount'], cash)
                    if amount > 0:
                        shares = amount / p
                        holdings[ticker] = {'shares': shares, 'max_price': p, 'entry_date': curr_date, 'entry_price': p}
                        cash -= amount
                        trade_log.append({
                            'Date': curr_date, 'Ticker': ticker, 'Name': ticker_to_name[ticker],
                            'Type': 'Buy', 'Price': p, 'Shares': shares,
                            'Reason': trade['reason'], 'Momentum_Value': trade.get('momentum', 0)
                        })
            pending_trades = []

        # 2. 更新權益與持有期間最高價
        port_value = cash
        for ticker, info in holdings.items():
            val = info['shares'] * curr_prices[ticker]
            port_value += val
            holdings[ticker]['max_price'] = max(holdings[ticker]['max_price'], curr_prices[ticker])
        equity.iloc[i] = port_value

        holdings_log.append({
            'Date': curr_date,
            'Holdings': {t: info['shares'] for t, info in holdings.items()},
            'Equity': port_value
        })

        # 3. 訊號生成 (T日訊號)
        if i == n_days - 1: continue

        # 每日停損檢查
        current_held_tickers = list(holdings.keys())
        for ticker in current_held_tickers:
            info = holdings[ticker]
            stop_triggered = False
            reason = ""
            if stop_loss_type == 'peak':
                if curr_prices[ticker] < info['max_price'] * (1 - stop_loss_val):
                    stop_triggered = True
                    reason = f"最高價回落停損 ({stop_loss_val*100:.1f}%)"
            elif stop_loss_type == 'ma':
                ma_days = int(stop_loss_val)
                if i >= ma_days:
                    ma_stop_val = prices[ticker].rolling(ma_days).mean().iloc[i]
                    if curr_prices[ticker] < ma_stop_val:
                        stop_triggered = True
                        reason = f"均線停損 ({ma_days})"

            if stop_triggered:
                if not any(t['ticker'] == ticker and t['type'] == 'sell' for t in pending_trades):
                    pending_trades.append({'ticker': ticker, 'type': 'sell', 'shares': info['shares'], 'reason': reason})

        # 定期再平衡
        if (i - start_idx) % rb_period == rb_offset:
            eligible = (prices.iloc[i] > sma.iloc[i]) & (roc.iloc[i] > 0)
            eligible_roc = roc.iloc[i][eligible].sort_values(ascending=False)
            top_3 = eligible_roc.head(3).index.tolist()

            # 賣出跌出排名的標的
            for ticker in list(holdings.keys()):
                if ticker not in top_3:
                    if not any(t['ticker'] == ticker and t['type'] == 'sell' for t in pending_trades):
                        pending_trades.append({'ticker': ticker, 'type': 'sell', 'shares': holdings[ticker]['shares'], 'reason': '跌出排名'})

            # 買入新進排名的標的 (份額保留原則)
            to_buy = [t for t in top_3 if t not in holdings and not any(tr['ticker'] == t and tr['type'] == 'buy' for tr in pending_trades)]
            if to_buy:
                filled_slots = len([t for t in top_3 if t in holdings and not any(tr['ticker'] == t and tr['type'] == 'sell' for tr in pending_trades)])
                open_slots = 3 - filled_slots
                if open_slots > 0:
                    amount_per_slot = port_value / 3
                    for ticker in to_buy[:open_slots]:
                        pending_trades.append({'ticker': ticker, 'type': 'buy', 'amount': amount_per_slot,
                                               'reason': f'ROC 排名進入前 3 ({roc.iloc[i][ticker]:.4f})',
                                               'momentum': roc.iloc[i][ticker]})

    equity = equity.ffill().fillna(initial_capital)
    return equity, trade_log, holdings_log

def calculate_metrics(equity):
    if len(equity) < 2 or equity.iloc[0] == 0: return {'CAGR': 0, 'MaxDD': 0, 'Calmar': 0}
    total_return = (equity.iloc[-1] / equity.iloc[0]) - 1
    years = (equity.index[-1] - equity.index[0]).days / 365.25
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 and total_return > -1 else -1
    drawdown = (equity / equity.cummax()) - 1
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if max_dd != 0 else 0
    return {
        'CAGR': cagr,
        'MaxDD': max_dd,
        'Calmar': calmar,
        'WinRate': (equity.pct_change() > 0).mean(),
        'TotalReturn': total_return
    }


## 3. 螞蟻演算法 (ACO) 參數最佳化
針對 SMA, ROC 以及停損參數進行搜尋。

In [ ]:

class ACO:
    def __init__(self, param_ranges, n_ants=10, n_iter=5, rho=0.1):
        self.param_ranges = param_ranges
        self.n_ants = n_ants
        self.n_iter = n_iter
        self.rho = rho
        self.pheromones = [np.ones(len(r)) for r in param_ranges]

    def run(self, fitness_func):
        best_p, best_f = None, -np.inf
        for t in range(self.n_iter):
            results = []
            for _ in range(self.n_ants):
                indices = [np.random.choice(len(r), p=p/p.sum()) for r, p in zip(self.param_ranges, self.pheromones)]
                values = [r[idx] for r, idx in zip(self.param_ranges, indices)]
                fit = fitness_func(values)
                results.append((indices, fit))
                if fit > best_f:
                    best_f = fit
                    best_p = values
            # 更新信息素
            for p in self.pheromones: p *= (1 - self.rho)
            for indices, fit in results:
                if fit > 0:
                    for i, idx in enumerate(indices):
                        self.pheromones[i][idx] += fit
            print(f"迭代 {t+1}/{self.n_iter}, 當前最佳適應度: {best_f:.4f}")
        return best_p, best_f

def perform_optimization(df_train):
    sma_range = [10, 20, 30, 40, 50, 60, 80, 100, 120, 150, 200]
    roc_range = [10, 20, 40, 60, 100, 120, 150, 200, 250]
    sl_peak_range = [0.02, 0.05, 0.08, 0.10]
    sl_ma_range = [3, 5, 10, 15]

    def fitness_peak(p):
        eq, _, _ = run_backtest(df_train, p[0], p[1], 'peak', p[2])
        return calculate_metrics(eq)['Calmar']

    def fitness_ma(p):
        eq, _, _ = run_backtest(df_train, p[0], p[1], 'ma', p[2])
        return calculate_metrics(eq)['Calmar']

    print("正在優化最高價回落停損參數...")
    bp, bf = ACO([sma_range, roc_range, sl_peak_range]).run(fitness_peak)
    print("正在優化均線停損參數...")
    bm, bmf = ACO([sma_range, roc_range, sl_ma_range]).run(fitness_ma)

    if bf >= bmf:
        return 'peak', bp
    else:
        return 'ma', bm


## 4. 執行優化與生成高原表

In [ ]:

# 使用最近 180 天進行優化
df_train = df.iloc[-180:]
sl_type, best_params = perform_optimization(df_train)
print(f"最佳停損類型: {sl_type}, 最佳參數 (SMA, ROC, SL): {best_params}")

# 生成高原表 (SMA vs Calmar, 固定 ROC 與 SL 為最佳值)
sma_range = [10, 20, 30, 40, 50, 60, 80, 100, 120, 150, 200]
plateau_results = []
for s in sma_range:
    eq, _, _ = run_backtest(df_train, s, best_params[1], sl_type, best_params[2])
    m = calculate_metrics(eq)
    plateau_results.append({
        'SMA': s,
        'CAGR': f"{m['CAGR']*100:.2f}%",
        'MaxDD': f"{m['MaxDD']*100:.2f}%",
        'Calmar': round(m['Calmar'], 2)
    })
plateau_df = pd.DataFrame(plateau_results)
plateau_df


## 5. 最終回測結果與視覺化

In [ ]:

equity, trades, h_log = run_backtest(df_train, best_params[0], best_params[1], sl_type, best_params[2])
metrics = calculate_metrics(equity)

plt.figure(figsize=(12, 6))
plt.plot(equity, label='策略權益曲線')
plt.fill_between(equity.index, equity.min(), equity, color='gray', alpha=0.1)
plt.title('Equity Curve (Optimization Period)')
plt.xlabel('Date')
plt.ylabel('Portfolio Value')
plt.grid(True)
plt.show()

for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")


## 6. 產出詳細 Excel 報告

In [ ]:

def export_results(equity, trades, h_log, metrics, params, sl_type, plateau_df, filename):
    # 準備 Trades
    trades_df = pd.DataFrame(trades)
    if not trades_df.empty:
        trades_df['SMA_Param'] = params[0]
        trades_df['ROC_Param'] = params[1]
        trades_df['StopLoss_Param'] = f"{sl_type}_{params[2]}"
        
        def get_desc(row):
            if row['Type'] == 'Buy':
                return f"選取動能值(ROC)為 {row['Momentum_Value']:.4f} 的資產 {row['Name']} 進場"
            else:
                return f"資產 {row['Name']} 因 {row['Reason']} 出場"
        trades_df['繁體中文說明'] = trades_df.apply(get_desc, axis=1)
    
    # 準備 Equity Curve
    equity_df = pd.DataFrame(equity, columns=['Equity'])
    equity_df['Drawdown'] = (equity_df['Equity'] / equity_df['Equity'].cummax()) - 1
    
    # 準備持股明細
    h_rows = []
    for entry in h_log:
        d = entry['Date']
        for t, s in entry['Holdings'].items():
            h_rows.append({'Date': d, 'Ticker': t, 'Shares': s})
    holdings_df = pd.DataFrame(h_rows)
    
    # 準備總結
    summary_data = {
        'Metric': ['CAGR', 'MaxDD', 'Calmar', 'WinRate', 'TotalReturn', 'Best_SMA', 'Best_ROC', 'Best_SL_Type', 'Best_SL_Val'],
        'Value': [
            f"{metrics['CAGR']*100:.2f}%",
            f"{metrics['MaxDD']*100:.2f}%",
            round(metrics['Calmar'], 2),
            f"{metrics['WinRate']*100:.2f}%",
            f"{metrics['TotalReturn']*100:.2f}%",
            params[0], params[1], sl_type, params[2]
        ]
    }
    summary_df = pd.DataFrame(summary_data)
    
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        trades_df.to_excel(writer, sheet_name='Trades', index=False)
        equity_df.to_excel(writer, sheet_name='Equity_Curve')
        holdings_df.to_excel(writer, sheet_name='Equity_Hold', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        plateau_df.to_excel(writer, sheet_name='Plateau', index=False)
    print(f"Excel 報告已產生: {filename}")

export_results(equity, trades, h_log, metrics, best_params, sl_type, plateau_df, 'trendstrategy_results_equity2024.xlsx')
